In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import InputLayer,Input
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import R2Score
SEED = 29

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
df = pd.read_parquet("train.parquet")

In [3]:
seqs = np.zeros((517,1000,32))
for i in range(517):
   seqs[i] = df[df["seq_ix"] == i].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).values

seqs = seqs.astype(np.float16)
# seqs[i][j][0] gives ith sequence jth row need_prediction


In [4]:
seqs.nbytes

33088000

In [5]:
window_size = 100
data = np.zeros((517*(1000-window_size-1),window_size,32),dtype=np.float16)
y_data = np.zeros((517*(1000-window_size-1),1,32),dtype=np.float16)

In [6]:
for i in range(len(data)):
    j = i//(1000-window_size-1)
    k = i% (1000-window_size-1)
    y_data[i] = seqs[j][k+window_size][:]
    #print(i)
    data[i] = seqs[j][k:k+window_size][:]

In [7]:
data.nbytes

2974611200

In [8]:
import tensorflow.keras.backend as K
def r2_keras(y_true, y_pred):
    """
    Custom R2-score metric for 3D tensors (batch, time_steps, features).
    Flattens the time steps dimension before calculating R2.
    """
    # Reshape tensors from (batch, time_steps, features) to (batch * time_steps, features)
    y_true_reshaped = K.reshape(y_true, (-1, K.shape(y_true)[-1]))
    y_pred_reshaped = K.reshape(y_pred, (-1, K.shape(y_pred)[-1]))

    # Calculate Sum of Squares Residual (SS_res)
    SS_res = K.sum(K.square(y_true_reshaped - y_pred_reshaped))

    # Calculate Total Sum of Squares (SS_tot)
    # Mean is calculated across all flattened samples
    SS_tot = K.sum(K.square(y_true_reshaped - K.mean(y_true_reshaped)))

    # Calculate R2 score, add epsilon to prevent division by zero
    r2 = 1 - SS_res / (SS_tot + K.epsilon())

    return r2

In [ ]:
def build_model(input_shape):
    num_neurons = [128,64]
    X = Input(input_shape,name ="X_input")
    enc_out = layers.LSTM(num_neurons = 40,return_sequences=True)
    trans_x = layers.Lambda(lambda x: tf.transpose(x, perm=[0, 2, 1]))(enc_out)
    out_dense_1 = layers.Dense(num_neurons[0],activation="relu")(trans_x)
    attn_output = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=num_neurons[0]//4)(out_dense_1, out_dense_1)

    # Add & Norm (Standard Transformer block)
    attn = tf.keras.layers.Add()([out_dense_1, attn_output])
    attn_norm = tf.keras.layers.LayerNormalization(axis=-1)(attn)

    # 4. FEED FORWARD NETWORK
    # This processes the information within each variable token
    ffn_1 = layers.Dense(num_neurons[0]*2, activation='relu')(attn_norm) 
    ffn_2 = layers.Dense(num_neurons[0])(ffn_1)
    ffn_combined = tf.keras.layers.Add()([attn_norm, ffn_2])
    ffn_norm = tf.keras.layers.LayerNormalization(axis=-1)(ffn_combined)

    x = tf.keras.layers.Dense(1)(ffn_norm) # Shape: (Batch, 32, 96)

    # 6. FINAL TRANSPOSE
    # Swap back to (Batch, Time, Variables) format
    outputs = layers.Lambda(lambda x: tf.transpose(x, perm=[0, 2, 1]))(x) # Shape: (Batch, 96, 32)

    model = tf.keras.Model(inputs=X, outputs=outputs)
    return model

    
    

In [10]:
model = build_model((100,32))

In [11]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ X_input             │ (None, 100, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 32, 100)   │          0 │ X_input[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32, 128)   │     12,928 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 32, 128)   │     66,048 │ dense[0][0],      │
│ (MultiHeadAttentio… │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 128)   │          0 │ dense[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 32, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32, 256)   │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32, 128)   │     32,896 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 128)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 128)   │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32, 1)     │        129 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 1, 32)     │          0 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 145,537 (568.50 KB)

 Trainable params: 145,537 (568.50 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_r2_keras',  
        factor=0.5,          
        patience=2,          
        verbose=1,           
        mode='max',         
        min_delta=0.002,      
        cooldown=2,          
        min_lr=0.0002        
    )
optim = tf.keras.optimizers.Adam(learning_rate=0.01,clipnorm = 1.0)
model.compile(optimizer=optim,loss="mse",metrics=[r2_keras])

In [ ]:
model.fit(data,y_data,epochs = 30,verbose = True,batch_size=32,validation_split = 0.2,callbacks = [reduce_lr])

Epoch 1/30
 6271/11620 ━━━━━━━━━━━━━━━━━━━━ 1:29 17ms/step - loss: 0.9931 - r2_keras: -0.0297